# Social app use for an ad budget: choices that overlap

**Hypothesis Testing for the Difference of Population Proportions for 1 Sample** · a worked example from the StatNex teaching guide

**The story.** An ad agency asked 50 students which social apps they use. Many use both a photo-sharing app and a short-video app. The agency must decide where to spend most of its budget.

**The question.** Is the photo-sharing app used by more students than the short-video app?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_proportion_diff__overlap.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Photo-sharing app,Short-video app
1.0,0.0,0.0
2.0,1.0,1.0
3.0,1.0,0.0
4.0,1.0,1.0
5.0,1.0,1.0
6.0,0.0,0.0
7.0,0.0,0.0
8.0,1.0,1.0
9.0,1.0,0.0
10.0,1.0,1.0
11.0,1.0,0.0
12.0,1.0,1.0
13.0,1.0,0.0
14.0,0.0,1.0
15.0,1.0,1.0
16.0,1.0,1.0
17.0,0.0,1.0
18.0,0.0,0.0
19.0,1.0,0.0
20.0,1.0,0.0
21.0,1.0,0.0
22.0,1.0,0.0
23.0,0.0,1.0
24.0,1.0,1.0
25.0,0.0,1.0
26.0,1.0,1.0
27.0,1.0,0.0
28.0,1.0,1.0
29.0,0.0,0.0
30.0,1.0,1.0
31.0,1.0,0.0
32.0,0.0,0.0
33.0,1.0,1.0
34.0,1.0,0.0
35.0,1.0,1.0
36.0,0.0,0.0
37.0,1.0,0.0
38.0,1.0,0.0
39.0,1.0,1.0
40.0,1.0,0.0
41.0,0.0,0.0
42.0,0.0,0.0
43.0,1.0,0.0
44.0,1.0,1.0
45.0,0.0,1.0
46.0,1.0,0.0
47.0,1.0,1.0
48.0,0.0,0.0
49.0,1.0,0.0
50.0,1.0,1.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Photo-sharing app", "Short-video app"]
confidence = 95 / 100
# Two columns coded 1 (yes) and 0 (no), one row per person. A row missing either
# answer says nothing about the pairing, so it is dropped.
answers = data[variables].apply(pd.to_numeric, errors="coerce").dropna()
first, second = answers["Photo-sharing app"], answers["Short-video app"]
total = len(answers)
n1, n2 = int(first.sum()), int(second.sum())
n12 = int(((first == 1) & (second == 1)).sum())
labels = variables
both = f"{labels[0]} and {labels[1]}"

## Counts and proportions

In [ ]:
counts_table = pd.DataFrame({"Total": [total], labels[0]: [n1], labels[1]: [n2]})
if n12 > 0:
    counts_table[both] = [n12]
counts_table.loc[1] = counts_table.loc[0] / total
counts_table.index = ["n", "proportion"]
print(counts_table)

## Confidence interval for the difference of proportions

In [ ]:
# The two proportions come from the same people, so the difference is judged by those
# in exactly one of the two categories: b = n1 - n12 and c = n2 - n12 people.
difference = n1 / total - n2 / total
discordant = n1 + n2 - 2 * n12
# Wald's interval, the textbook one: the variance of the difference at the observed
# proportions, (b + c - (b - c)² / n) / n² (R's PropCIs::diffpropci.Wald.mp gives the
# same). With few people in only one category, or a share near 0 or 1, Tango's score
# interval covers better.
se = np.sqrt(discordant - (n1 - n2) ** 2 / total) / total
z_critical = stats.norm.ppf(0.5 + confidence / 2)
interval = pd.DataFrame([[difference, difference - z_critical * se, difference + z_critical * se]],
                        columns=["difference", "lower", "upper"])
print()
print(interval.to_string(index=False))

## Test for the difference of proportions (one sample)

In [ ]:
# This is McNemar's test. Lay the answers out as a 2 x 2 table (both, only 1 / only 2,
# neither): if the two categories were equally common, the people in exactly one of
# them would split evenly between the two. z = (n1 - n2) / sqrt(n1 + n2 - 2 n12) keeps
# the direction, and z squared is McNemar's chi-square without continuity correction,
# which statsmodels' mcnemar() computes and turns into the two-sided P. The normal
# approximation needs about 10 or more people in exactly one category; with fewer,
# mcnemar(..., exact=True) gives the exact binomial P instead.
paired_table = np.array([[n12, n1 - n12], [n2 - n12, total - n1 - n2 + n12]])
z = (n1 - n2) / np.sqrt(discordant)
p_two = mcnemar(paired_table, exact=False, correction=False).pvalue
# Use the one-sided P only if its direction was fixed before seeing the data.
p_one = p_two / 2
test = pd.DataFrame([[z, p_two, stars(p_two), p_one, stars(p_one)]],
                    columns=["z", "P two-sided", "", "P one-sided", ""])
print()
print(test.to_string(index=False))

## What the result says

Of the 50 students, 35 use the photo-sharing app (a proportion of 0.7) and 23 use the short-video app (0.46). Here 18 students (0.36) use both, so the two groups overlap. The test takes this overlap into account.

The difference of proportions is 0.24. Its 95% confidence interval, the range of values the data agree with for the true difference, runs from 0.07 to 0.41 and does not include zero. The z statistic is 2.56 and the two-sided P value is 0.011. The P value is the chance of a gap at least this large if both apps were really used equally.

So the photo-sharing app is used by more students. But many students use both apps, and for an ad budget that matters as much as which app is bigger.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.